# Chest X-ray — SMALL run (C3 go/no-go)

Trains 3 diverse models on REAL data, runs the full pipeline with **live progress**, and prints the **§8b verdict**. GO -> use colab_full.ipynb.

Progress streams live because cells use the `!` shell magic (not buffered subprocess).

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone / update code from GitHub

In [ ]:
import os
REPO = 'https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    !git clone $REPO /content/Chest_Xray
else:
    !git -C /content/Chest_Xray pull
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — unzip data to fast local disk (slow, once per session)
Live unzip progress shown per file.

In [ ]:
import os, glob
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('unzipping', os.path.basename(z), flush=True)
    !unzip -q -o "$z" -d data/raw
print('folders:', os.listdir('data/raw'))

## Cell 4 — bring the manifest

In [ ]:
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE_DIR}/manifest.csv'):
    shutil.copy(f'{DRIVE_DIR}/manifest.csv','data/manifest.csv'); print('manifest copied')
else:
    !python scripts/build_manifest.py

## Cell 5 — install deps + GPU check (GPU must NOT be empty)
pip progress streams live.

In [ ]:
!pip -q install -r requirements.txt
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 6 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df = pd.read_csv('data/manifest.csv')
missing = [p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
if missing:
    print('EXAMPLE:', missing[0]); print('folders:', os.listdir('data/raw'))
    print('>>> STOP: fix folder mapping before running.')
else:
    print('>>> OK: paths resolve. Continue.')

## Cell 7 — FULL pipeline on 3 models (LIVE progress; ~20-40 min)
You will see: per-stage banners, live Keras epoch bars during training, per-model audit/XAI counters.

In [ ]:
MODELS = 'densenet201 resnet50 vit'
!python -u scripts/run_pipeline.py --models $MODELS --epochs 20 --batch-size 64

## Cell 8 — READ THE §8b VERDICT (decide threshold BEFORE looking)

In [ ]:
import json
c = json.load(open('results/c3_coupling.json'))
print(json.dumps(c, indent=2))
if c.get('n_models',0) >= 3 and 'delta_acc' in c:
    r2 = c['delta_acc']['r2']; r = c['delta_acc']['r']
    print(f"\nSRC -> delta_acc:  r={r:+.3f}  R2={r2:.3f}")
    print('GO -> colab_full.ipynb (spine = METHOD)' if (r>0 and r2>=0.3)
          else 'WEAK/NO-GO -> spine = AUDIT; rethink SRC (PAPER_OUTLINE.md §8b)')
else:
    print('Need 3 models with valid SRC + cross-source results.')

## Cell 9 — SAVE results to Drive (Colab wipes /content at session end)

In [ ]:
import datetime
stamp = datetime.date.today().isoformat()
!cp -r results /content/drive/MyDrive/cxr_data/results_small_$stamp
print('saved -> Drive/cxr_data/results_small_'+stamp)